# Visium HD demo data: data-driven analysis buffer (paper section 2)

All public 10x Visium HD datasets processed with "v1 - Protocol 2.0" that have an H&E image
(11 datasets), at 8 µm. Included: the 8 human samples, all tissue blocks or cores whose lateral
edges were cut (trimmed surgical tissue, TMA punches). Excluded (`dataset_type = visium_demo_excluded`):
mouse brain, mouse embryo (whole-organ / whole-organism sections, natural boundaries where edge effects
and anatomy, e.g. cell-poor cortical layer 1, cannot be separated) and rat liver (lobe piece, capsule
on part of the boundary).

**Border = tissue edge.** Analysed bins: raw 8 µm bins inside the image-only tissue mask (Otsu on the
hires H&E) and inside the capture area, n_counts > 0 (holes <= 1024 µm² filled, components > 1000 bins).
Each bin's distance is to the edge of the **whole** image mask: where the capture area cuts through
tissue that continues, the cut is not a border (the effect is physical, at the tissue's own edge).
The same mask defines
"outside" for the diffusion fit (section 3).

**Border effect** = piecewise-linear elbow per component, fitted on bins within 1 mm of the tissue edge,
constrained to rise into the tissue and then plateau (m >= 0, `bosperrus.RisingPiecewiseLinearFit`),
winning against a constant by AIC, and passing the
border criterion: an empirical p-value <= 0.05 against interior windows (section below; no physical
border, fitted exactly like the tissue edges), on bosperrus' own fit descriptors -- observed effect strength
and half-life (µm from the border to the midpoint of the rise). The test statistic is the number of windows
at least as deep and at least as narrow, calibrated against the same count among the windows
(`src/spatial_omics/border_criterion.py`); nothing is tuned on the tissue edges. BLADE (as published) before and after bosperrus correction is the benchmark.

Supplements at the end: an 8 vs. 2 µm resolution check of the elbow, and the same panels with
Space Ranger's `in_tissue` bins. The paired Visium HD / Stereo-seq comparison (section 3) is in
`ST_paired.ipynb`.

Everything is read from `results/spatial_omics/st_reroll/`, written on a compute node by
`src/spatial_omics/st_precompute.py` (`jobs/spatial_omics_st_precompute.sbatch <sample>`, then
`<sample> --image-mask`, `--in-tissue`, `--tissue-edge --resolution {8,2}`) and
`src/spatial_omics/interior_windows.py`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from bosperrus import FIT_PALETTE, plot_fit
from mpl_toolkits.axes_grid1 import make_axes_locatable
import sys
sys.path.insert(0, "../../src/spatial_omics")
from border_criterion import ALPHA, criterion_metrics, false_positive_rate, load_window_null, passes, window_p_value

# Everything below reads precomputed per-sample results -- written on a compute
# node by src/spatial_omics/st_precompute.py (see st_precompute.sbatch), which runs
# bosperrus.load_filtered + identify_analysis_buffer_from_filtered (+ BLADE) and
# bosperrus.quantify_diffusion_from_raw at 8um for every sample in st_samples.csv.
RESULTS_DIR = Path("../../results/spatial_omics/st_reroll")
SAMPLES = pd.read_csv("../../src/spatial_omics/st_samples.csv")

SAMPLES = SAMPLES[SAMPLES["dataset_type"] == "visium_demo"].reset_index(drop=True)
print(len(SAMPLES), "demo samples:", ", ".join(SAMPLES["name"]))

In [ ]:
DATASET_TYPE_MARKERS = {"visium_demo": "o", "visium_custom": "s", "stomics_custom": "^", "Visium HD": "s", "STOmics": "^"}

# Tissue edge: elbow per component and BLADE before/after correction (main);
# % of spots in the buffer and elbow vs. BLADE (supplement)

All rising elbows (AIC vs. constant, rising-only fit) are shown; the border criterion is not applied
here; filled markers pass it, open markers do not (false-positive rate on interior windows in the
title). A component without a rising elbow is drawn at
"none" and has 0% of its bins in the buffer. Uncorrected BLADE peels from the analysed bins' own edge
(capture-area cuts included); corrected BLADE uses the tissue-edge fit.


In [ ]:
def _nan(v):
    return np.nan if v is None else v  # JSON null (e.g. ConstantFit won / BLADE never reached p >= 0.05)


WINDOW_DIR = RESULTS_DIR / "interior_windows"
NULL = load_window_null(WINDOW_DIR, SAMPLES["name"])  # 8 um interior windows of the included demo samples


def load_buffer_results(suffix, null=NULL):
    """Per-component results from {sample}_buffer{suffix}.json ("_imagemask": bins inside the image
    mask, distance to the tissue edge; "_intissue": Space Ranger's in_tissue bins), with the border
    criterion (border_criterion.py: p-value of effect strength and half-life against the interior windows
    `null`, d_max from the per-bin parquet) in columns `p_value` and `passes`. Every rising elbow
    (AIC + m > 0) is kept."""
    rows = []
    for sample in SAMPLES["name"]:
        buf = json.loads((RESULTS_DIR / f"{sample}_buffer{suffix}.json").read_text())
        bin_size_um = buf["resolution_um"]
        bins = pd.read_parquet(RESULTS_DIR / f"{sample}_filtered_bins{suffix}.parquet",
                               columns=["components", "virtual", "distance_to_border"])
        bins = bins[~bins["virtual"]]
        n_bins = bins.groupby("components").size()  # whole component, also bins beyond the fit range
        d_max = bins[np.isfinite(bins["distance_to_border"])].groupby("components")["distance_to_border"].max()
        for label, info in buf["per_component"].items():
            # bsp_thresh_um: piecewise_linear_b, already in um; bsp_thresh: the same in grid steps
            bsp_thresh_um = _nan(info["bsp_thresh_um"])
            eff, half = criterion_metrics(info, d_max[int(label)])
            p = window_p_value(eff, half, null)
            ok = passes(p)
            rows.append([
                sample, int(label), f"{sample}_{label}", n_bins[int(label)],
                _nan(info["blade_thresh"]), bsp_thresh_um / bin_size_um, _nan(info["blade_thresh_corrected"]),
                bin_size_um,
                _nan(info["blade_thresh_um"]), bsp_thresh_um, _nan(info["blade_thresh_corrected_um"]),
                info["pct_affected"], eff, half, p, ok,
            ])
    df = pd.DataFrame(rows, columns=[
        "sample", "component_rank", "component_label", "component_size",
        "blade_thresh", "bsp_thresh", "blade_thresh_corrected",
        "bin_size_um", "blade_thresh_um", "bsp_thresh_um", "blade_thresh_corrected_um",
        "pct_affected", "effect_strength", "half_life_um", "p_value", "passes",
    ])
    df = df.merge(SAMPLES[["name", "dataset_type"]], left_on="sample", right_on="name")
    df["component_size_um"] = df["component_size"] * df["bin_size_um"] ** 2
    df["sample_x"] = df["sample"].map({s: i for i, s in enumerate(sorted(df["sample"].unique()))})  # bin count -> physical area (um^2)

    # x positions for panel c: components side by side, a wider gap between samples
    df = df.sort_values(["sample", "component_rank"]).reset_index(drop=True)
    step = np.where(df["sample"] != df["sample"].shift(), 2, 1)
    step[0] = 0
    df["x_pos"] = np.cumsum(step)
    return df


results_df = load_buffer_results("_imagemask")

# one colour per demo sample (same in main figure and supplement)
sample_pal = dict(zip(SAMPLES["name"], sns.color_palette("tab20", len(SAMPLES))))


In [ ]:
def plot_blade_vs_bosperrus(ax, df, title, zero_pos=2.0, sep=5.0):
    """Panel b: bosperrus elbow vs. BLADE threshold on log-log axes. A missing
    buffer -- no border effect (elbow None/NaN) or BLADE = 0 -- is drawn at a
    stand-in position labelled "0" (zero_pos, left of/below the smallest real
    values, ~9um elbow / 8um BLADE), separated from them by dotted lines at sep."""
    d = df.assign(bsp_plot=df["bsp_thresh_um"].fillna(zero_pos),
                  blade_plot=df["blade_thresh_um"].replace(0, zero_pos))
    scatter_by_criterion(ax, d, "bsp_plot", "blade_plot")
    max_val = np.nanmax(df[["blade_thresh_um", "bsp_thresh_um"]].values)
    ax.plot((sep, max_val), (sep, max_val), c="gray", ls="-")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.axvline(sep, color="gray", ls=":", lw=0.8)
    ax.axhline(sep, color="gray", ls=":", lw=0.8)
    ticks = [zero_pos] + [t for t in [10, 100, 1000] if t <= max_val * 1.5]
    labels = ["0"] + [f"$10^{{{int(np.log10(t))}}}$" for t in ticks[1:]]
    ax.set_xticks(ticks, labels)
    ax.set_yticks(ticks, labels)
    ax.minorticks_off()
    ax.set_xlim(zero_pos / 1.6, max_val * 1.6)
    ax.set_ylim(zero_pos / 1.6, max_val * 1.6)
    ax.set_title(title)
    ax.set_ylabel("BLADE threshold ($\\mu m$)")
    ax.set_xlabel("bosperrus elbow ($\\mu m$)")


SAMPLE_ORDER = sorted(results_df["sample"].unique())  # x positions of the per-sample panels
WINDOW_FP = f"false-positive rate on {len(NULL['count'])} interior windows: {false_positive_rate(NULL):.1%}"


def scatter_by_criterion(ax, df, x, y):
    """Components passing the border criterion filled, all others open (same colour/size mapping)."""
    kw = dict(x=x, y=y, style="dataset_type", hue="sample", size="component_size_um", markers=DATASET_TYPE_MARKERS,
              sizes=(10, 500), size_norm=(df["component_size_um"].min(), df["component_size_um"].max()),
              palette=sample_pal, ax=ax, legend=False)
    sns.scatterplot(df[df["passes"]], **kw)
    if (~df["passes"]).any():
        sns.scatterplot(df[~df["passes"]], **kw)
        coll = ax.collections[-1]
        coll.set_edgecolors(coll.get_facecolors())
        coll.set_facecolors("none")
        coll.set_linewidths(1.2)


def _sample_xticks(ax):
    ax.set_xticks(range(len(SAMPLE_ORDER)), SAMPLE_ORDER, rotation=30, ha="right")
    ax.set_xlim(-0.5, len(SAMPLE_ORDER) - 0.5)


def _log_axis_with_none(ax, axis, max_val, zero_pos=2.0, sep=5.0):
    """Log axis with a stand-in position labelled "none" (zero_pos) below the smallest real values."""
    ticks = [zero_pos] + [t for t in [10, 100, 1000] if t <= max_val * 1.5]
    labels = ["none"] + [f"$10^{{{int(np.log10(t))}}}$" for t in ticks[1:]]
    getattr(ax, f"set_{axis}scale")("log")
    getattr(ax, f"set_{axis}ticks")(ticks, labels)
    getattr(ax, f"set_{axis}lim")(zero_pos / 1.6, max_val * 1.6)
    (ax.axhline if axis == "y" else ax.axvline)(sep, color="gray", ls=":", lw=0.8)
    ax.minorticks_off()


def plot_elbow(ax, df, title, zero_pos=2.0):
    """Elbow (um) per component, grouped by sample; no border effect drawn at "none"."""
    d = df.assign(bsp_plot=df["bsp_thresh_um"].fillna(zero_pos))
    scatter_by_criterion(ax, d, "sample_x", "bsp_plot")
    _log_axis_with_none(ax, "y", np.nanmax(df["bsp_thresh_um"]), zero_pos)
    ax.set_ylabel("bosperrus elbow ($\\mu m$)")
    ax.set_xlabel("")
    ax.set_title(title)
    _sample_xticks(ax)


def plot_pct_affected(ax, df, title):
    scatter_by_criterion(ax, df, "sample_x", "pct_affected")
    ax.set_yscale("symlog", linthresh=1)  # log above 1%, linear below so components without effect (0%) stay visible
    ax.set_ylim(-0.1, 100)
    ax.set_ylabel("% of spots within bosperrus buffer")
    ax.set_xlabel("")
    ax.set_title(title)
    _sample_xticks(ax)


def plot_correction(ax, df, title):
    """BLADE threshold before (plain) and after (outlined) bosperrus correction, per component."""
    for _, row in df.iterrows():
        ax.plot([row["x_pos"], row["x_pos"]], [row["blade_thresh_um"], row["blade_thresh_corrected_um"]], color="gray", linewidth=1, zorder=0)
    sns.scatterplot(data=df, x="x_pos", y="blade_thresh_um", hue="sample", style="dataset_type", markers=DATASET_TYPE_MARKERS, palette=sample_pal, sizes=(10, 500), size="component_size_um", edgecolor=None, legend=True, ax=ax, zorder=2)
    sns.scatterplot(data=df, x="x_pos", y="blade_thresh_corrected_um", hue="sample", style="dataset_type", markers=DATASET_TYPE_MARKERS, palette=sample_pal, sizes=(10, 500), size="component_size_um", edgecolor=FIT_PALETTE["Exponential Saturation Fit"], linewidth=1.5, legend=False, ax=ax, zorder=2)
    ax.set_xticks(df["x_pos"])
    ax.set_xticklabels(df["component_label"], rotation=90, fontsize=6)
    ax.set_xlabel("")
    ax.set_ylabel("BLADE threshold ($\\mu m$)")
    ax.set_title(title)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))


def save(save_as):
    plt.tight_layout()
    Path("../../result_plots/spatial_omics").mkdir(parents=True, exist_ok=True)
    plt.savefig(f"../../result_plots/spatial_omics/{save_as}", bbox_inches="tight")
    plt.show()


def plot_main(df, border, save_as):
    """Main figure: a = elbow per sample, b = BLADE before/after correction."""
    f, axs = plt.subplots(1, 2, figsize=(14, 4.5), width_ratios=[1.3, 3])
    plot_elbow(axs[0], df, f"Demo data, {border}:\nbosperrus elbow\n(filled: p <= {ALPHA} vs. interior windows;\n{WINDOW_FP})")
    plot_correction(axs[1], df, f"{border}: BLADE thresholds before\nand after bosperrus correction")
    save(save_as)


def plot_supp(df, border, save_as):
    """Supplement: % of spots in the buffer, elbow vs. BLADE."""
    f, axs = plt.subplots(1, 2, figsize=(9, 4.5), width_ratios=[1.3, 1])
    plot_pct_affected(axs[0], df, f"Demo data, {border}:\nPercentage of affected spots\n(filled: p <= {ALPHA} vs. interior windows)")
    plot_blade_vs_bosperrus(axs[1], df, f"Demo data, {border}:\nbosperrus vs. BLADE")
    save(save_as)


print(results_df[["component_label", "bsp_thresh_um", "effect_strength", "half_life_um", "pct_affected"]].round(1).to_string())
print(results_df.groupby("sample")[["pct_affected", "bsp_thresh_um", "blade_thresh_um"]].median().round(1))
plot_main(results_df, "tissue edge", "ST_panels_abc.svg")
plot_supp(results_df, "tissue edge", "ST_panels_supp_tissue_edge.svg")


# Null of the border criterion: interior windows vs. tissue edges

Interior windows (`src/spatial_omics/interior_windows.py`, `results/spatial_omics/st_reroll/interior_windows/`):
squares of 0.5/1/2 mm inside the capture area, >= 500 µm from any tissue edge, 20 per size and sample,
fit exactly like the tissue edges (rising-only fit within 1 mm) with distance to
the window edge. No physical border, so every elbow
there is a false positive. The TMA cores are too small for windows.

In [ ]:
windows = pd.concat([pd.read_csv(f) for f in sorted(WINDOW_DIR.glob("*_windows.csv")) if f.stat().st_size > 1])
windows = windows[windows["sample"].isin(SAMPLES["name"])]  # included demo samples only
print(f"{len(windows)} windows: elbow by AIC + m > 0 in {windows['border_effect'].mean():.1%}; "
      f"false-positive rate of the criterion (leave-one-out): {false_positive_rate(NULL):.1%}")
print(windows.groupby("window_um")["border_effect"].mean().round(3).rename("rising elbow").to_string())

edge = results_df.dropna(subset=["effect_strength"])
rising = windows[windows["border_effect"]]
f, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(rising["half_life_um"], rising["effect_strength"], c="lightgray", s=12,
           label=f"interior windows, rising fit ({len(rising)}/{len(windows)})")
for ok, fill in [(True, True), (False, False)]:
    e = edge[edge["passes"] == ok]
    ax.scatter(e["half_life_um"], e["effect_strength"], s=np.sqrt(e["component_size_um"]) / 25,
               c=[sample_pal[s] for s in e["sample"]] if fill else "none",
               edgecolors=[sample_pal[s] for s in e["sample"]], linewidths=1.2,
               label=f"tissue edges, p <= {ALPHA}" if fill else f"tissue edges, p > {ALPHA}")
ax.set_xscale("log")
ax.set_xlabel("half-life ($\\mu m$ from the border to the midpoint of the rise)")
ax.set_ylabel("observed effect strength")
ax.set_title(f"Tissue edges (colour) vs. interior windows (grey)\n"
             f"false-positive rate on the windows: {false_positive_rate(NULL):.1%} at p <= {ALPHA}")
ax.legend(fontsize=7, loc="lower right")
plt.tight_layout()
plt.savefig("../../result_plots/spatial_omics/ST_criterion_calibration.svg", bbox_inches="tight")
plt.show()


# Diffusion outside the image mask, per demo sample

Pooled, component-free fit (ConstantFit vs. ExponentialDecayFit) on every raw 8 µm bin outside the
image-only H&E mask, zero-count bins included (`bosperrus.quantify_diffusion_from_raw`). The
cross-technology comparison of these fits is in `ST_paired.ipynb`.

In [ ]:
raw_pooled_df = pd.DataFrame([
    json.loads((RESULTS_DIR / f"{sample}_diffusion_8um.json").read_text()) for sample in SAMPLES["name"]
])
raw_pooled_df["1/beta"] = raw_pooled_df["decay_length_um"]
raw_pooled_df[["sample", "best_fit_type", "alpha", "beta", "1/beta", "perc_counts_outside", "mask_tissue_fraction", "n_bins_total"]]

In [ ]:
# One bosperrus.plot_fit per sample: the pooled diffusion fit (ConstantFit vs.
# ExponentialDecayFit on every raw 8um bin outside the mask, zero-count bins
# included -- the fit behind panels e/f) over the per-distance-bin quantile bands and mean.
#
# The curve uses the fitted per-bin amplitude a (params["exponential_decay_a"]),
# not alpha: alpha = a / bin area is the cross-resolution-comparable number, but
# the plot shows counts per bin.
#
# Shared distance bin edges (log-spaced) across all subplots. n_counts+1 so zero-count
# bins (most bins far from the tissue) stay visible on a log axis.
all_distance, all_counts = [], []
for sample in raw_pooled_df["sample"]:
    pts = np.load(RESULTS_DIR / f"{sample}_diffusion_8um.npz")
    outside = pts["distance_um"] > 0
    all_distance.append(pts["distance_um"][outside])
    all_counts.append(pts["n_counts"][outside])
all_distance, all_counts = np.concatenate(all_distance), np.concatenate(all_counts) + 1

N_BINS = 60
x_edges = np.logspace(np.log10(all_distance.min()), np.log10(all_distance.max()), N_BINS + 1)
del all_distance, all_counts

N_COLS = 4
N_ROWS = int(np.ceil(len(raw_pooled_df) / N_COLS))
fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(15, 5.3 * N_ROWS), sharex=True, sharey=True, squeeze=False)
for ax in axes.flat[len(raw_pooled_df):]:
    ax.set_visible(False)
for ax, (_, row) in zip(axes.flat, raw_pooled_df.iterrows()):
    pts = np.load(RESULTS_DIR / f"{row['sample']}_diffusion_8um.npz")
    distance, n_counts = pts["distance_um"], pts["n_counts"]
    outside = distance > 0

    params = row["params"]
    if row["best_fit_type"] == "Exponential Decay Fit":
        a, b = params["exponential_decay_a"], params["exponential_decay_b"]
        predict_fn = lambda d, a=a, b=b: a * np.exp(-b * d) + 1  # +1 to match the n_counts+1 shift below
    else:  # ConstantFit baseline won instead
        c = params["constant_c"] + 1
        predict_fn = lambda d, c=c: np.full_like(d, c)

    plot_fit(ax, distance[outside], n_counts[outside] + 1, predict_fn, bins=x_edges,
             line_kwargs={"color": FIT_PALETTE.get(row["best_fit_type"], "red")})
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"{row['sample']} ({row['dataset_type']})", fontsize=9)
    ax.set_xlabel("distance outside mask (um)")
    ax.set_ylabel("n_counts + 1 (raw, per 8um bin)")
    ax.set_ylim(bottom=0.8)  # log axis: n_counts + 1 >= 1
plt.tight_layout()
plt.show()

# Supplement: resolution check, panel a at 8 µm vs. 2 µm

Is the narrow tissue-edge dip a physical rim (same width in µm at both bin sizes) or bin geometry
(1-2 bins wide, i.e. narrower at 2 µm)? Panel a only, identical settings at both bin sizes
(`st_precompute.py <sample> --tissue-edge --resolution {8,2}`): every bin inside the image mask, zero-count
bins kept (at 2 µm many tissue bins have no counts), rising-only elbow vs. distance to the tissue edge within
1 mm, minimum component size scaled to the same area; no BLADE. Interior windows with the same setup at each bin size
are the null of the criterion at that bin size (`interior_windows.py --resolution {8,2}`). The image mask itself is at the hires H&E resolution
(2.1-5.2 µm per pixel), so the tissue edge is only defined to about that precision.


In [ ]:
RESOLUTIONS = [8, 2]


def load_tissue_edge(res, null):
    """Per-component panel-a results of the --tissue-edge run at `res` um, with the border criterion."""
    rows = []
    for sample in SAMPLES["name"]:
        buf = json.loads((RESULTS_DIR / f"{sample}_buffer_tissueedge_{res}um.json").read_text())
        bins = pd.read_parquet(RESULTS_DIR / f"{sample}_filtered_bins_tissueedge_{res}um.parquet",
                               columns=["array_row", "array_col", "components"])
        centre = bins[bins["components"] >= 0].groupby("components")[["array_row", "array_col"]].mean() * res
        for label, info in buf["per_component"].items():
            eff, half = criterion_metrics(info, info["max_distance_um"])
            p = window_p_value(eff, half, null)
            rows.append({"sample": sample, "component_rank": int(label), "component_label": f"{sample}_{label}",
                         "resolution_um": res, "bsp_thresh_um": _nan(info["elbow_um"]),
                         "pct_affected": info["pct_affected"], "effect_strength": eff, "half_life_um": half,
                         "p_value": p, "passes": passes(p), "dataset_type": "visium_demo",
                         "component_size_um": info["n_bins"] * res ** 2,
                         "centre_row_um": centre.loc[int(label), "array_row"],
                         "centre_col_um": centre.loc[int(label), "array_col"]})
    df = pd.DataFrame(rows)
    df["sample_x"] = df["sample"].map({s: i for i, s in enumerate(SAMPLE_ORDER)})
    return df


null_res = {res: load_window_null(WINDOW_DIR, SAMPLES["name"], f"_tissueedge_{res}um") for res in RESOLUTIONS}
edge_res = {res: load_tissue_edge(res, null_res[res]) for res in RESOLUTIONS}
for res in RESOLUTIONS:
    d = edge_res[res]
    print(f"{res} um: {d['passes'].sum()}/{len(d)} components pass; false-positive rate on "
          f"{len(null_res[res]['count'])} interior windows {false_positive_rate(null_res[res]):.1%}")

# match components between bin sizes by centre position (same capture area, grids nested)
from scipy.spatial import cKDTree
a, b = edge_res[8], edge_res[2]
matched = []
for sample in SAMPLES["name"]:
    a_s, b_s = a[a["sample"] == sample], b[b["sample"] == sample]
    dist, j = cKDTree(b_s[["centre_row_um", "centre_col_um"]]).query(a_s[["centre_row_um", "centre_col_um"]])
    for (_, r8), jj, dd in zip(a_s.iterrows(), j, dist):
        r2 = b_s.iloc[jj]
        matched.append({"sample": sample, "component_label": r8["component_label"], "centre_offset_um": dd,
                        "half_8": r8["half_life_um"], "half_2": r2["half_life_um"],
                        "elbow_8": r8["bsp_thresh_um"], "elbow_2": r2["bsp_thresh_um"],
                        "pass_8": r8["passes"], "pass_2": r2["passes"],
                        "component_size_um": r8["component_size_um"]})
matched = pd.DataFrame(matched)
print(f"matched {len(matched)} components (centre offset max {matched['centre_offset_um'].max():.0f} um); "
      f"pass at 8 um {matched['pass_8'].sum()}, at 2 um {matched['pass_2'].sum()}, both {(matched['pass_8'] & matched['pass_2']).sum()}")
both = matched[matched["pass_8"] & matched["pass_2"]]
print(f"half-life of components passing at both: median {both['half_8'].median():.1f} um at 8 um, "
      f"{both['half_2'].median():.1f} um at 2 um (ratio 2/8 median {(both['half_2'] / both['half_8']).median():.2f})")
print(matched.round(1).to_string())


In [ ]:
def edge_profiles(sample, res, max_um=300.0, step_um=4.0):
    """Mean counts per um^2 vs. distance to the tissue edge (all components pooled), normalised to the
    mean between 500 um and 1 mm."""
    bins = pd.read_parquet(RESULTS_DIR / f"{sample}_filtered_bins_tissueedge_{res}um.parquet",
                           columns=["n_counts", "components", "distance_full_um"])
    bins = bins[bins["components"] >= 0]
    density = bins["n_counts"] / res ** 2
    plateau = density[(bins["distance_full_um"] >= 500) & (bins["distance_full_um"] <= 1000)].mean()
    edges = np.arange(0, max_um + step_um, max(step_um, res))
    step = pd.cut(bins["distance_full_um"], edges, right=False)
    prof = density.groupby(step, observed=True).mean() / plateau
    return np.array([iv.mid for iv in prof.index]), prof.to_numpy()


f, axs = plt.subplots(1, 4, figsize=(20, 4.5), width_ratios=[1.3, 1.3, 1, 1.3])
for ax, res in zip(axs[:2], RESOLUTIONS):
    plot_elbow(ax, edge_res[res], f"tissue edge, {res} $\\mu m$ bins: bosperrus elbow\n"
               f"(filled: p <= {ALPHA}; false-positive rate on windows {false_positive_rate(null_res[res]):.1%})")
ax = axs[2]
m = matched.dropna(subset=["half_8", "half_2"])
for ok, fill in [(True, True), (False, False)]:
    mm = m[(m["pass_8"] & m["pass_2"]) == ok]
    ax.scatter(mm["half_8"], mm["half_2"], s=np.sqrt(mm["component_size_um"]) / 30,
               c=[sample_pal[s] for s in mm["sample"]] if fill else "none",
               edgecolors=[sample_pal[s] for s in mm["sample"]], linewidths=1.2)
lim = (1, max(m["half_8"].max(), m["half_2"].max()) * 1.5)
ax.plot(lim, lim, color="gray")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("half-life at 8 $\\mu m$ ($\\mu m$)"); ax.set_ylabel("half-life at 2 $\\mu m$ ($\\mu m$)")
ax.set_title("same component, both bin sizes\n(filled: passes at both)")
ax = axs[3]
for sample in SAMPLES["name"]:
    for res, ls in [(8, "--"), (2, "-")]:
        x, y = edge_profiles(sample, res)
        ax.plot(x, y, color=sample_pal[sample], ls=ls, lw=1)
ax.axhline(1, color="gray", lw=0.8)
ax.set_xlim(0, 300); ax.set_ylim(bottom=0)
ax.set_xlabel("distance to tissue edge ($\\mu m$)"); ax.set_ylabel("counts per $\\mu m^2$ / plateau (0.5-1 mm)")
ax.set_title("edge profiles, all components pooled\n(solid: 2 $\\mu m$, dashed: 8 $\\mu m$)")
save("ST_resolution_check.svg")


# Supplement: panels a-c with Space Ranger's `in_tissue` bins as tissue footprint

Same analysis as the main figure (fit within 1 mm of the edge, same border criterion), but the bins are
Space Ranger's filtered (`in_tissue`) bins (`bosperrus.load_filtered`) and the border is the edge of their
footprint, so capture-area cuts count as border (`st_precompute.py <sample> --in-tissue`,
`*_buffer_intissue.json`).

In [ ]:
spaceranger_df = load_buffer_results("_intissue")
print(spaceranger_df[["component_label", "bsp_thresh_um", "effect_strength", "half_life_um", "pct_affected"]].round(1).to_string())
print(spaceranger_df.groupby("sample")[["pct_affected", "bsp_thresh_um", "blade_thresh_um"]].median().round(1))
plot_main(spaceranger_df, "Space Ranger in_tissue", "ST_panels_abc_spaceranger.svg")
plot_supp(spaceranger_df, "Space Ranger in_tissue", "ST_panels_supp_spaceranger.svg")
